# Open Higgs: Kaggle twin T4 workers

Two independent ComfyUI processes, one per T4. This increases throughput for independent shots. It does not pool VRAM, shard one model, or guarantee that every imported workflow fits a T4.

In Kaggle notebook settings choose **GPU T4 x2** and enable **Internet**. Add the Kaggle secret `OPENHIGGS_WORKER_TOKEN` with the same value as your studio server. The studio must be reachable over HTTPS. A localhost URL on your laptop is not reachable from Kaggle.

Expose only `/api/studio/worker/*` through an HTTPS reverse proxy. Keep the normal studio interface on loopback. See `docs/OPEN_HIGGS.md` for the proxy example. Notebook runtime and GPU availability remain subject to Kaggle limits.

The baseline installs ComfyUI at a pinned revision and downloads SD Turbo. Read its model license before downloading or commercial use: https://huggingface.co/stabilityai/sd-turbo . Imported video and editing graphs require their own weights and custom nodes, installed on both workers.


In [ ]:
import os, sys, pathlib, subprocess, time, json, urllib.request, shutil
import torch
assert torch.cuda.is_available(), 'Select a GPU accelerator in notebook settings.'
assert torch.cuda.device_count() >= 2, 'Select GPU T4 x2.'
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, round(p.total_memory / 2**30, 1), 'GiB')
BASE = pathlib.Path('/kaggle/working/openhiggs')
BASE.mkdir(parents=True, exist_ok=True)
COMFY_REVISION = 'fb2b26fdac507b7da81277c7becaf54c5e46ac6b'
COMFY = BASE / 'ComfyUI'
if not COMFY.exists():
    subprocess.run(['git','clone','--filter=blob:none','https://github.com/Comfy-Org/ComfyUI.git',str(COMFY)],check=True)
subprocess.run(['git','-C',str(COMFY),'checkout',COMFY_REVISION],check=True)
VENV = BASE / 'venv'
if not (VENV / 'bin/python').exists():
    subprocess.run([sys.executable,'-m','venv','--system-site-packages',str(VENV)],check=True)
PYTHON = str(VENV / 'bin/python')
subprocess.run([PYTHON,'-m','pip','install','-r',str(COMFY/'requirements.txt')],check=True)


## Download the baseline checkpoint

This downloads several GB into the shared models directory. Both GPU processes reuse the file. You can instead attach a Kaggle dataset containing the checkpoint and copy or symlink it here. The starter workflow is `open-source-canvas → sd turbo` in the studio library.


In [ ]:
checkpoint = COMFY / 'models/checkpoints/sd_turbo.safetensors'
checkpoint.parent.mkdir(parents=True, exist_ok=True)
if not checkpoint.exists():
    partial = checkpoint.with_suffix('.partial')
    url = 'https://huggingface.co/stabilityai/sd-turbo/resolve/b261bac6fd2cf515557d5d0707481eafa0485ec2/sd_turbo.safetensors'
    try:
        urllib.request.urlretrieve(url, partial)
        partial.replace(checkpoint)
    except Exception:
        partial.unlink(missing_ok=True)
        raise
print('Checkpoint ready:', checkpoint.name)


## Connect your studio

Enter only the HTTPS worker endpoint origin below, for example `https://gpu-bridge.example.com`. The token is read from Kaggle Secrets and is never printed. The embedded worker is the same script bundled with the studio. Each job has a lease and heartbeats; interrupted jobs are not silently resubmitted.


In [ ]:
from kaggle_secrets import UserSecretsClient
STUDIO_URL = input('Studio HTTPS origin: ').strip().rstrip('/')
assert STUDIO_URL.startswith('https://'), 'Use an HTTPS endpoint reachable from Kaggle.'
TOKEN = UserSecretsClient().get_secret('OPENHIGGS_WORKER_TOKEN')
assert len(TOKEN) >= 24, 'Use the same worker token as the studio (24+ characters).'


In [ ]:
WORKER_SOURCE = '"""Outbound-only ComfyUI worker, one instance per GPU. Python standard library only."""\nimport argparse\nimport base64\nimport json\nimport os\nimport re\nimport threading\nimport time\nimport urllib.error\nimport urllib.parse\nimport urllib.request\nimport uuid\n\n\ndef prepare_assets(studio, comfy, token, job):\n    graph = job[\'payload\'][\'workflow\']\n    for asset in job[\'payload\'].get(\'assets\', []):\n        req = urllib.request.Request(studio + \'/api/studio/worker/assets/\' + asset[\'id\'], headers={\'Authorization\': \'Bearer \' + token})\n        with urllib.request.urlopen(req, timeout=90) as response:\n            data = response.read(45_000_000 + 1)\n        if len(data) > 45_000_000:\n            raise RuntimeError(\'Input media exceeds the 45 MB worker limit.\')\n        boundary = \'OpenHiggs\' + uuid.uuid4().hex\n        name = re.sub(r\'[^\\w.-]\', \'_\', asset[\'name\'], flags=re.ASCII)\n        body = (f\'--{boundary}\\r\\nContent-Disposition: form-data; name="image"; filename="{name}"\\r\\nContent-Type: application/octet-stream\\r\\n\\r\\n\'.encode() + data + f\'\\r\\n--{boundary}--\\r\\n\'.encode())\n        req = urllib.request.Request(comfy + \'/upload/image\',data=body,headers={\'Content-Type\':\'multipart/form-data; boundary=\'+boundary},method=\'POST\')\n        with urllib.request.urlopen(req,timeout=90) as response:\n            result=json.load(response)\n        if not result.get(\'name\'):\n            raise RuntimeError(\'ComfyUI did not return an uploaded filename.\')\n        graph[asset[\'node\']][\'inputs\'][asset[\'field\']]=\'/\'.join(part for part in [result.get(\'subfolder\'),result[\'name\']] if part)\n    return graph\n\n\ndef request(url, payload=None, headers=None, method=None):\n    data = json.dumps(payload).encode() if payload is not None else None\n    req = urllib.request.Request(url, data=data, headers={\'Content-Type\': \'application/json\', **(headers or {})}, method=method)\n    with urllib.request.urlopen(req, timeout=45) as response:\n        return json.load(response)\n\n\ndef process_job(studio, comfy, token, job):\n    headers = {\'Authorization\': \'Bearer \' + token}\n    endpoint = studio + \'/api/studio/worker/\' + job[\'id\']\n    stop = threading.Event()\n    lost_lease = threading.Event()\n\n    def heartbeat():\n        while not stop.wait(20):\n            try:\n                request(endpoint, {\'lease\': job[\'lease\'], \'status\': \'running\'}, headers)\n            except urllib.error.HTTPError as error:\n                if error.code == 409:\n                    lost_lease.set()\n                    return\n            except (OSError, ValueError):\n                pass\n\n    beat = threading.Thread(target=heartbeat, daemon=True)\n    beat.start()\n    try:\n        # Never retry a submission with an unknown outcome.\n        graph = prepare_assets(studio, comfy, token, job)\n        task = request(comfy + \'/prompt\', {\'prompt\': graph, \'client_id\': job[\'id\']})\n        if task.get(\'error\') or task.get(\'node_errors\'):\n            raise RuntimeError(\'ComfyUI rejected workflow: \' + json.dumps(task)[:1000])\n        prompt_id = task[\'prompt_id\']\n        while True:\n            if lost_lease.is_set():\n                raise RuntimeError(\'Studio lease lost; inspect this notebook before submitting a replacement.\')\n            try:\n                history = request(comfy + \'/history/\' + prompt_id).get(prompt_id)\n            except (OSError, ValueError):\n                time.sleep(5)\n                continue\n            if history and history.get(\'status\', {}).get(\'status_str\') == \'error\':\n                raise RuntimeError(json.dumps(history[\'status\'].get(\'messages\', []))[:1500])\n            if history and history.get(\'status\', {}).get(\'completed\'):\n                break\n            time.sleep(3)\n        outputs = []\n        for output in history.get(\'outputs\', {}).values():\n            for kind in [\'images\', \'gifs\', \'videos\', \'audio\', \'files\']:\n                for file in output.get(kind, []):\n                    if not file.get(\'filename\'):\n                        continue\n                    query = urllib.parse.urlencode({key: file.get(key, default) for key, default in [(\'filename\', \'\'), (\'subfolder\', \'\'), (\'type\', \'output\')]})\n                    with urllib.request.urlopen(comfy + \'/view?\' + query, timeout=90) as response:\n                        data = response.read(45_000_000 + 1)\n                    if len(data) > 45_000_000:\n                        raise RuntimeError(\'Output exceeds the 45 MB transport limit. Save it from Kaggle and lower the preview size.\')\n                    name = str(len(outputs)) + \'-\' + re.sub(r\'[^\\w.-]\', \'_\', os.path.basename(file[\'filename\']), flags=re.ASCII)\n                    upload = request(endpoint + \'/artifact/\' + urllib.parse.quote(name), {\'base64\': base64.b64encode(data).decode()}, {**headers, \'X-Worker-Lease\': job[\'lease\']}, \'PUT\')\n                    outputs.append(upload[\'url\'])\n        if not outputs:\n            raise RuntimeError(\'Workflow completed without supported outputs.\')\n        request(endpoint, {\'lease\': job[\'lease\'], \'status\': \'completed\', \'outputs\': outputs}, headers)\n    except Exception as error:\n        try:\n            request(endpoint, {\'lease\': job[\'lease\'], \'status\': \'failed\', \'error\': str(error)}, headers)\n        except Exception:\n            print(\'Could not report job result. Job will expire without automatic retry.\', flush=True)\n        print(\'Job failed:\', str(error), flush=True)\n    finally:\n        stop.set()\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--studio\', required=True)\n    parser.add_argument(\'--comfy\', default=\'http://127.0.0.1:8188\')\n    parser.add_argument(\'--worker\', default=\'kaggle-t4-0\')\n    args = parser.parse_args()\n    token = os.environ.get(\'OPENHIGGS_WORKER_TOKEN\', \'\')\n    if len(token) < 24:\n        raise SystemExit(\'Set OPENHIGGS_WORKER_TOKEN to a shared secret of at least 24 characters.\')\n    if not args.studio.startswith(\'https://\') and not args.studio.startswith((\'http://127.0.0.1:\', \'http://localhost:\')):\n        raise SystemExit(\'A remote studio URL must use HTTPS.\')\n    studio, comfy = args.studio.rstrip(\'/\'), args.comfy.rstrip(\'/\')\n    print(\'Worker ready:\', args.worker, flush=True)\n    while True:\n        try:\n            claim = request(studio + \'/api/studio/worker/claim\', {\'worker\': args.worker}, {\'Authorization\': \'Bearer \' + token})\n            if claim.get(\'job\'):\n                process_job(studio, comfy, token, claim[\'job\'])\n            else:\n                time.sleep(4)\n        except urllib.error.HTTPError as error:\n            if error.code in (401, 403):\n                raise SystemExit(\'Worker authentication or access rejected. Check studio URL and matching worker token.\')\n            print(\'Studio temporarily unavailable:\', error.code, flush=True)\n            time.sleep(10)\n        except (OSError, ValueError) as error:\n            print(\'Connection unavailable:\', type(error).__name__, flush=True)\n            time.sleep(10)\n\n\nif __name__ == \'__main__\':\n    main()\n'
worker_file = BASE / 'kaggle_worker.py'
worker_file.write_text(WORKER_SOURCE)


In [ ]:
# Stop processes from a previous run of this cell before starting replacements.
for process in globals().get('PROCESSES', []):
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
PROCESSES = []
PORTS = [8188, 8190]
for gpu, port in enumerate(PORTS):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    log = (BASE / f'comfy-{gpu}.log').open('w')
    process = subprocess.Popen([PYTHON,str(COMFY/'main.py'),'--listen','127.0.0.1','--port',str(port),'--output-directory',str(BASE/f'outputs-{gpu}'),'--temp-directory',str(BASE/f'temp-{gpu}'),'--force-fp16','--use-pytorch-cross-attention','--disable-auto-launch'],cwd=COMFY,env=env,stdout=log,stderr=subprocess.STDOUT)
    PROCESSES.append(process)
    for attempt in range(180):
        if process.poll() is not None:
            raise RuntimeError(f'ComfyUI GPU {gpu} exited. Read {BASE}/comfy-{gpu}.log')
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:{port}/system_stats',timeout=3) as response:
                stats=json.load(response)
            break
        except Exception:
            time.sleep(2)
    else:
        raise RuntimeError(f'ComfyUI GPU {gpu} did not become ready.')
    env['OPENHIGGS_WORKER_TOKEN'] = TOKEN
    worker_log = (BASE / f'worker-{gpu}.log').open('w')
    process = subprocess.Popen([PYTHON,str(worker_file),'--studio',STUDIO_URL,'--comfy',f'http://127.0.0.1:{port}','--worker',f'kaggle-t4-{gpu}'],env=env,stdout=worker_log,stderr=subprocess.STDOUT)
    PROCESSES.append(process)
    print(f'GPU {gpu}: ComfyUI ready, worker started.')
print('In Open Higgs choose Twin T4 and queue two independent shots.')


## Check workers and stop

Large video models need quantization, CPU offload, smaller resolution or different hardware. T4 should use FP16; do not assume BF16/FP8 kernels are supported. The included setup validates connectivity only when you run it. A successful studio build does not prove GPU inference.


In [ ]:
for gpu in range(2):
    path = BASE / f'worker-{gpu}.log'
    print(f'Worker {gpu}:')
    print(path.read_text()[-2500:] if path.exists() else 'Not started')
print('Process states:', [p.poll() for p in PROCESSES])


In [ ]:
# Set True and run this cell when finished. Outputs uploaded to the studio survive shutdown.
STOP_WORKERS = False
if STOP_WORKERS:
    for process in PROCESSES:
        if process.poll() is None:
            process.terminate()
